<a href="https://colab.research.google.com/github/DataFriend101/Machine_Learning/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
# Load the database
%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [9]:
# Build the 90-day table from the warehouse (for simplicity)
df = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_date
    FROM {TABLES['fact_daily']}
),
windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.report_date,
        f.gsc_impressions,
        f.gsc_clicks,
        f.gsc_avg_position,
        CASE
            WHEN f.report_date <= b.end_date - INTERVAL 45 DAY
            THEN 'first_half' ELSE 'second_half'
        END AS half
    FROM {TABLES['fact_daily']} f
    CROSS JOIN bounds b
    WHERE f.report_date > b.end_date - INTERVAL 90 DAY
      AND f.client_has_gsc = TRUE
      AND f.gsc_data_available = TRUE
)
SELECT
    client_hash_id,
    content_hash_id AS content_id,
    SUM(gsc_impressions) AS impressions_90d,
    SUM(gsc_clicks) AS clicks_90d,
    AVG(gsc_avg_position) AS avg_position,
    SUM(CASE WHEN half = 'first_half'  THEN gsc_clicks END) AS clicks_first_half,
    SUM(CASE WHEN half = 'second_half' THEN gsc_clicks END) AS clicks_second_half
FROM windowed
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) > 0
""").df()

print(f"Rows: {len(df):,}")
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 270,645


,client_hash_id,content_id,impressions_90d,clicks_90d,avg_position,clicks_first_half,clicks_second_half
0,client_e547b89c05043229,content_1557a3abbc832229,500.0,2.0,17.742535,0.0,2.0
1,client_e547b89c05043229,content_e1f6d0c859ba9dc4,749.0,0.0,23.821131,0.0,0.0
2,client_e547b89c05043229,content_48537762b74f5b34,751.0,0.0,23.079575,0.0,0.0
3,client_e547b89c05043229,content_27b27b5e13d4e6b7,558.0,2.0,21.367975,1.0,1.0
4,client_e547b89c05043229,content_8c2c3dab1f1e875f,1838.0,3.0,25.493199,2.0,1.0


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I prioritize content that already has search visibility but may have room for improvement. Pages with many impressions and weaker click engagement are ranked higher because improving pages that already receive attention could create a larger impact.

Reason codes:

- HIGH_VISIBILITY_REFRESH: The page receives strong search visibility and may be worth reviewing for improvement.
- LOW_CTR_OPPORTUNITY: The page receives impressions but has weaker click performance.
- REVIEW: The page does not strongly match the main opportunity signals but may still be worth checking.

In [10]:
# Signal 1: Volume
df["impression_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    duplicates="drop"
)
volume_check = (
    df.groupby("impression_bucket", observed=True)
    .agg(
        n=("content_id", "count"),
        avg_clicks=("clicks_90d", "mean")
    )
    .reset_index()
)

volume_check

,impression_bucket,n,avg_clicks
0,"(0.999, 16.0]",68510,0.022843
1,"(16.0, 168.0]",66851,0.359007
2,"(168.0, 1252.0]",67632,1.536758
3,"(1252.0, 1961517.0]",67652,41.641001


#### Volume signal verdict: **Confirmed**

Average clicks rise sharply across impression buckets from 0.02 in the lowest quartile to 41.6 in the highest. This suggests that pages with stronger visibility have more clicks




In [11]:
# Signal 2: CTR opportunity
df["ctr"] = (df["clicks_90d"] / df["impressions_90d"]).fillna(0)

df["ctr_bucket"] = pd.qcut(
    df["ctr"],
    q=4,
    duplicates="drop"
)

ctr_check = (
    df.groupby("ctr_bucket", observed=True)
    .agg(
        n=("content_id", "count"),
        avg_impressions=("impressions_90d", "mean"),
        avg_clicks=("clicks_90d", "mean")
    )
    .reset_index()
)

ctr_check

,ctr_bucket,n,avg_impressions,avg_clicks
0,"(-0.001, 0.00286]",202988,2141.792865,2.907512
1,"(0.00286, 1.0]",67657,4870.307729,34.828710


#### CTR signal verdict: **Mixed**

Some pages have strong visibility but weaker CTR, which may indicate optimization opportunities. However, CTR alone does not explain whether the content itself needs improvement

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

To create the ranked queue, I combine the signals from my rule into a simple score. The score is designed to prioritize pages that already have visibility but may have room for improvement.

A higher score means the page may be a stronger candidate for review. The score is only used for prioritization and is not a prediction of future performance.

In [12]:
# Normalize visibility so larger pages receive higher scores
df["visibility_score"] = df["impressions_90d"] / df["impressions_90d"].max()

# Pages with lower CTR relative to the dataset receive a higher opportunity score
df["ctr_opportunity"] = 1 - (df["ctr"] / df["ctr"].max())

# Score = visibility_score * ctr_opportunity
df["score"] = df["visibility_score"] * df["ctr_opportunity"]
df["score"].describe()

,score
count,270645.000000
mean,0.001434
std,0.006554
min,0.000000
25%,0.000008
50%,0.000086
75%,0.000636
max,0.992238


In [13]:
# Define simple thresholds
high_visibility_threshold = df["impressions_90d"].quantile(0.75)
low_ctr_threshold = df["ctr"].median()

def assign_reason(row):
    if row["impressions_90d"] >= high_visibility_threshold and row["ctr"] <= low_ctr_threshold:
        return "LOW_CTR_OPPORTUNITY"
    elif row["impressions_90d"] >= high_visibility_threshold:
        return "HIGH_VISIBILITY_REFRESH"
    else:
        return "REVIEW"

df["reason_code"] = df.apply(assign_reason, axis=1)
df["reason_code"].value_counts()

,count
reason_code,
REVIEW,202963
HIGH_VISIBILITY_REFRESH,63998
LOW_CTR_OPPORTUNITY,3684


In [14]:
def assign_action(reason):
    if reason == "LOW_CTR_OPPORTUNITY":
        return "OPTIMIZE_SNIPPET"
    elif reason == "HIGH_VISIBILITY_REFRESH":
        return "REVIEW_UPDATE"
    return "MONITOR"

df["action"] = df["reason_code"].apply(assign_action)
df["action"].value_counts()

,count
action,
MONITOR,202963
REVIEW_UPDATE,63998
OPTIMIZE_SNIPPET,3684


In [17]:
import numpy as np

DECLINE_THRESHOLD_PCT = -20

df["trend_pct"] = (
    (df["clicks_second_half"] - df["clicks_first_half"])
    / df["clicks_first_half"].replace(0, np.nan)
) * 100
df["trend_pct"] = df["trend_pct"].fillna(0)
df["is_declining_label"] = (df["trend_pct"] <= DECLINE_THRESHOLD_PCT).astype(int)

print(df["is_declining_label"].value_counts())
print(f"Base rate: {df['is_declining_label'].mean():.3f}")

is_declining_label
0    227944
1     42701
Name: count, dtype: int64
Base rate: 0.158


In [18]:
# create ranked queue
baseline_queue = (
    df[["content_id", "client_hash_id", "score", "reason_code", "action", "is_declining_label"]]
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

baseline_queue.head(10)

,content_id,client_hash_id,score,reason_code,action,is_declining_label
0,content_eadb33b5df496f4a,client_e547b89c05043229,0.992238,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,1
1,content_545bb6cc7081ded3,client_e547b89c05043229,0.499491,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,0
2,content_963de14b1f58978f,client_e547b89c05043229,0.460083,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,0
3,content_943dc881428182b8,client_8ddc46da5414ffd8,0.346228,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,0
4,content_acbcc847f8996314,client_62f4a7e64f5e0096,0.307975,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,0
5,content_21309e9a83c83653,client_e547b89c05043229,0.301882,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,0
6,content_f107e54b10b43725,client_62f4a7e64f5e0096,0.288138,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,1
7,content_33d31496fca9665e,client_73cda7b4e4f265ea,0.281554,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,1
8,content_0e03de7680314cd5,client_e547b89c05043229,0.281285,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,1
9,content_62770e1299963fe4,client_73cda7b4e4f265ea,0.266234,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,0


In [19]:
# precision@K
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df["is_declining_label"].mean()
for k in [20, 50, 100]:
    p = precision_at_k(df["score"], df["is_declining_label"], k)
    print(f"precision@{k}: {p:.3f}   (base rate: {base_rate:.3f})")

precision@20: 0.500   (base rate: 0.158)
precision@50: 0.520   (base rate: 0.158)
precision@100: 0.520   (base rate: 0.158)


In [20]:
# Create the CSV
import os
os.makedirs("work/outputs", exist_ok=True)
baseline_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [25]:
baseline_queue.head(20)

,content_id,client_hash_id,score,reason_code,action,is_declining_label
0,content_eadb33b5df496f4a,client_e547b89c05043229,0.992238,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,1
1,content_545bb6cc7081ded3,client_e547b89c05043229,0.499491,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,0
2,content_963de14b1f58978f,client_e547b89c05043229,0.460083,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,0
3,content_943dc881428182b8,client_8ddc46da5414ffd8,0.346228,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,0
4,content_acbcc847f8996314,client_62f4a7e64f5e0096,0.307975,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,0
5,content_21309e9a83c83653,client_e547b89c05043229,0.301882,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,0
6,content_f107e54b10b43725,client_62f4a7e64f5e0096,0.288138,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,1
7,content_33d31496fca9665e,client_73cda7b4e4f265ea,0.281554,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,1
8,content_0e03de7680314cd5,client_e547b89c05043229,0.281285,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,1
9,content_62770e1299963fe4,client_73cda7b4e4f265ea,0.266234,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE,0


All 20 of the top-ranked pages are HIGH_VISIBILITY_REFRESH.LOW_CTR_OPPORTUNITY doesn't appear until lower in the queue, which means at the **top of the list the score is effectively driven by visibility alone**, the CTR-opportunity term isn't yet differentiating the highest-priority pages.

HIGH_VISIBILITY_REFRESH → REVIEW_UPDATE: **Confidence medium** since 10 of the top 20 are actually labeled declining so roughly half of these picks would be "wrong" by the label, meaning a reviewer following this queue blind would waste time on about half their reviews at this rank.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some of the weaker picks are pages selected mainly because they have high visibility with no confirmation the content itself is the problem: precision@20 = 0.500 means half the top picks are false positives.

### Leakage check
I confirmed that the score only uses historical performance features:
- impressions_90d
- clicks_90d
- calculated CTR

Rows are also filtered to `client_has_gsc = TRUE` and `gsc_data_available = TRUE`, so missing data isn't counted as zero performance.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.